# Experiment

The 3 layer version of the offense defense model had high variance. I wonder what the performance could look like on the 2 layer model instead. This should no longer have high variance

In [1]:
import sys, os
sys.path.insert(0, "/home/roman/Code/AIEngineering/Projects/FIFACompetition/WorldCup2026/experiments")

In [2]:
# Imports
# Allowing notebook to import from components folder
import sys, os
root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.insert(0, root_path)
    
import numpy as np
from components.player_modification import arrange_player_details_to_offense_defense
from components.data import load_original_data_with_trimmed_y, collect_player_data_into_sum_concat_teams_form_from_old, split_and_normalize_dataset
from components.constants import alpha
from models.v2 import ModelV2

# Loading Data

Load and normalize offense defense data

In [3]:
X, Y = load_original_data_with_trimmed_y()
X_offense_defense = collect_player_data_into_sum_concat_teams_form_from_old(
    Input=X,
    num_player_features=8,
    extract_player_vector=arrange_player_details_to_offense_defense
)

X_train, X_cv, X_test, Y_train, Y_cv, Y_test = split_and_normalize_dataset(
    X=X_offense_defense,
    Y=Y,
    training_set_ratio=0.86,
    cv_test_set_ratio=0.5,
    random_state=121
)

print(X_train.shape, Y_train.shape)
print(X_cv.shape, Y_cv.shape)
print(X_test.shape, Y_test.shape)

(16, 1203) (121, 1203)
(16, 98) (121, 98)
(16, 99) (121, 99)


# Train

Train the model

In [4]:
import numpy as np
import numpy.typing as npt

class ModelV2:
    """ 
    Version of model that uses 2 layers
    """
    def __init__(
        self,
        W1_shape: tuple[int, int],
        b1_shape: tuple[int, int],
        W2_shape: tuple[int, int],
        b2_shape: tuple[int, int],
        seed: int
    ):
        """ 
        This function defines the shape of the parameters used by the model. From this it can initialize weights
        
        Args:
            W1_shape (tuple): tuple with shape of weights for first layer
            b1_shape (tuple): tuple with shape of biases for first layer
            W2_shape (tuple): tuple with shape of weights for second layer
            b2_shape (tuple): tuple with shapes of biases for second layer
            seed (scalar): seed used to generate random numbers
        """
        assert W1_shape[0] == b1_shape[0], "Weights and biases for first layer should have the same number of units"
        assert W2_shape[0] == b2_shape[0], "Weights and biases for second layer should have same number of units"
        assert W2_shape[1] == W1_shape[0], "Weights of second layer should accept same number of inputs as units of first layer"
        
        self.W1_shape = W1_shape
        self.b1_shape = b1_shape
        self.W2_shape = W2_shape
        self.b2_shape = b2_shape
        self.rng = np.random.default_rng(seed=seed)
        
    def initialize_weights(self):
        """ 
        Initialize weights with the indicated shape
        
        Return:
            W1 (ndarray): initial weights for first layer
            b1 (ndarray): initial bias for first layer
            W2 (ndarray): initial weights for second layer
            b2 (ndarray): initial bias for second layer
        """
        initial_W1 = self.rng.random(self.W1_shape) * np.sqrt(1 / self.W1_shape[1])
        initial_b1 = np.zeros(self.b1_shape)
        initial_W2 = self.rng.random(self.W2_shape) * np.sqrt(1 / self.W2_shape[1])
        initial_b2 = np.zeros(self.b2_shape)
        
        return (initial_W1, initial_b1, initial_W2, initial_b2)
    
    # Vectorized version of forward pass
    def f_x(
        X: npt.NDArray,
        W1: npt.NDArray,
        b1: npt.NDArray,
        W2: npt.NDArray,
        b2: npt.NDArray
    ):
        """ 
        Vectorized implementation of forward propagation
        
        Args:
            X (ndarray): array with m training examples
            W1 (ndarray): array with weights of first layer
            b1 (ndarray): array with biases of first layer
            W2 (ndarray): array with weights of second layer
            b2 (ndarray): array with biases of second layer
            
        Returns:
            y_pred (ndarray): a (121, m) array with predicted game outcomes
            Z_1 (ndarray): cached z_1 value
            A_1 (ndarray): cached a_1 value
        """
        print(W1.shape, X.shape, b1.shape)
        Z_1 = np.matmul(W1, X) + b1
        print(Z_1.shape)
        A_1 = np.maximum(0, Z_1)
        print(A_1.shape, W2.shape, b2.shape)
        Z_2 = np.matmul(W2, A_1) + b2
        print(Z_2.shape)
        
        shifted_logits = Z_2 - np.max(Z_2, axis=1, keepdims=True)
        e_z = np.exp(shifted_logits)
        Y_pred = e_z / np.sum(e_z, axis=1, keepdims=True)
        print(Y_pred.shape)
        
        return (Y_pred, Z_1, A_1)

    def J(
        Y_pred: npt.NDArray,
        Y: npt.NDArray,
    ):
        """ 
        Return the cost J given concatenated parameters theta
        
        Args:
            Y_pred (ndarray): a (121, m) array with predicted labels
            Y (ndarray) : a (121, m) with target labels
            
        Returns:
            J (scalar): cost
        """
        m = Y.shape[1]
        epsilon = 1e-15
        pred_clipped = np.clip(Y_pred, epsilon, 1 - epsilon)
        step_1 = Y * np.log(pred_clipped)
        example_losses = -np.sum(step_1, axis=0)
        average_loss = np.sum(example_losses) / m
        
        return average_loss

    def backprop(
        X: npt.NDArray,
        Y: npt.NDArray,
        W1: npt.NDArray,
        b1: npt.NDArray,
        W2: npt.NDArray,
        b2: npt.NDArray
    ):
        """ 
        Get derivatives that will be used by gradient descent to minimize cost function
        
        Args:
            X (ndarray): array with m training examples
            Y (ndarray): array with m training output classes
            W1 (ndarray): array with weights of first layer
            b1 (ndarray): array with bias of first layer
            W2 (ndarray): array with weights of second layer
            b2 (ndaray): array with bias of second layer
            
        Returns:
            dW1 (ndarray): derivative of J with respect to W1
            db1 (ndarray): derivative of J with respect to b1
            dW2 (ndarray): derivative of J with respect to W2
            db2 (ndarray): derivative of J with respect to b2
        """
        print("Starting backprop")
        m = X.shape[1]
        Y_pred, Z1, A1 = ModelV2.f_x(X=X, W1=W1, W2=W2, b1=b1, b2=b2)
        print("Got prediction z1 and a1 in backprop")
        
        
        dZ2 = Y_pred - Y
        dW2 = np.matmul(dZ2, A1.T) / m
        db2 = np.sum(dZ2, axis=1) / m
        dA1 = np.matmul(W2.T, dZ2)
        dZ1 = np.where(Z1 < 0, np.zeros(Z1.shape), dA1)
        dW1 = np.matmul(dZ1, X.T) / m
        db1 = np.sum(dZ1, axis=1) / m
        
        print("finished backprop")
        return (dW1, db1, dW2, db2)
        
    def train(
        self,
        X: npt.NDArray, 
        W1: npt.NDArray,
        b1: npt.NDArray,
        W2: npt.NDArray,
        b2: npt.NDArray,
        Y: npt.NDArray,
        alpha: float,
        num_iters: int,
        J_n: int = 10
    ):
        """ 
        This function will minimize the cost of the model using gradient descent and return found weights
        
        Args:
            X (ndarray) - a (m, 22, 9) array serving as input to model
            W1 (ndarray): array with weights of first layer
            b1 (ndarray): array with bias of first layer
            W2 (ndarray): array with weights of second layer
            b2 (ndaray): array with bias of second layer
            Y (ndarray) - a (m, 2, 1) array with target outputs for model
            alpha (scalar) - learning rate
            num_iters (scalar) - number of times to run gradient descent
            J_n (scalar) - interval to store training example
            
        Return:
            W1 (ndarray) - updated weights of first layer
            W2 (scalar) - updated weights of second layer
            b1 (ndarray) - updated bias of first layer
            b2 (scalar) - updated bias of second layer
            Js (list) - list of training costs and epoch at different iterations of training
        """
        Js = []
        
        for epoch in range(num_iters):
            dW1, db1, dW2, db2 = ModelV2.backprop(X=X, Y=Y, W1=W1, b1=b1, W2=W2, b2=b2)
            
            # Update weights
            W1 = W1 - (alpha * dW1)
            W2 = W2 - (alpha * dW2)
            b1 = b1 - (alpha * db1)
            b2 = b2 - (alpha * db2)
            
            
            Y_pred, _, _ = ModelV2.f_x(X=X, W1=W1, b1=b1, W2=W2, b2=b2)
            print(f"f_x result shape {Y_pred.shape}")
            cost = ModelV2.J(Y_pred=Y_pred, Y=Y)
            
            # Print result
            print(f"Cost at epoch {epoch}/{num_iters} = {cost}")
            
            if epoch % J_n == 0 or epoch == num_iters - 1:
                Js.append((epoch, cost))
            
        return (W1, W2, b1, b2, Js)

In [5]:
model_v2 = ModelV2(W1_shape=(64,16), b1_shape=(64,1), W2_shape=(121, 64), b2_shape=(121,1), seed=56)
init_W1, init_b1, init_W2, init_b2 = model_v2.initialize_weights()

updated_W1, updated_W2, updated_b1, updated_b2, Js = model_v2.train(
    X=X_train, 
    W1=init_W1,
    b1=init_b1,
    W2=init_W2,
    b2=init_b2,
    Y=Y_train,
    alpha=alpha,
    num_iters=1000
)

Starting backprop
(64, 16) (16, 1203) (64, 1)
(64, 1203)
(64, 1203) (121, 64) (121, 1)
(121, 1203)
(121, 1203)
Got prediction z1 and a1 in backprop
finished backprop
(64, 16) (16, 1203) (64, 64)


ValueError: operands could not be broadcast together with shapes (64,1203) (64,64) 